In [131]:
import matplotlib.pyplot as plt
%matplotlib inline 

words = open('names.txt', 'r').read().splitlines()

In [132]:
import torch

In [133]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0

itos = {i:s for s, i in stoi.items()}

In [149]:
N = torch.zeros((27, 27), dtype = torch.int32)

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        N[stoi[ch1], stoi[ch2]] += 1

P = N.float() + 1
P /= P.sum(1, keepdim = True)


In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range (5):
    ix = 0
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[ix])
        if (ix == 0):
            break
    print(''.join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


In [151]:
log_likelihood = 0.0
n=0
for w in ["andrejq"]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n+=1
        print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')
    
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

.a: 0.1376 -1.9835
an: 0.1604 -1.8302
nd: 0.0384 -3.2594
dr: 0.0770 -2.5646
re: 0.1334 -2.0143
ej: 0.0027 -5.9004
jq: 0.0003 -7.9817
q.: 0.0970 -2.3331
nll=tensor(27.8672)
3.4834020137786865


In [188]:
### Neural net version
xs = []
ys = []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [189]:
#one hot encoding
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes = 27).float()
yenc = F.one_hot(ys, num_classes = 27).float()

g = torch.Generator().manual_seed(10291920190)
W = torch.randn((27, 27), generator = g, requires_grad = True)
num = xs.nelement()


In [ ]:
for i in range (1000):
    matrix = xenc @ W
    cnt = matrix.exp()
    prob = cnt / cnt.sum(1, keepdims = True)
    loss = -prob[torch.arange(num), ys].log().mean()
    print(loss.item())
    W.grad = None
    loss.backward()
    W.data += -50 * W.grad



2.4612021446228027
2.4611668586730957
2.461132049560547
2.461097478866577
2.4610633850097656
2.461029291152954
2.460995674133301
2.4609622955322266
2.4609291553497314
2.4608964920043945
2.4608638286590576
2.460831642150879
2.4607994556427
2.460767984390259
2.4607365131378174
2.460705280303955
2.4606740474700928
2.4606435298919678
2.460613250732422
2.460582733154297
2.4605531692504883
2.4605233669281006
2.460493803024292
2.4604647159576416
2.460435628890991
2.460407257080078
2.460378408432007
2.460350275039673
2.460322380065918
2.460294723510742
2.460266590118408
2.4602396488189697
2.460212469100952
2.4601855278015137
2.4601588249206543
2.460132122039795
2.460106134414673
2.4600799083709717
2.4600541591644287
2.4600284099578857
2.460003137588501
2.459977388381958
2.4599523544311523
2.459927558898926
2.459902763366699
2.459878444671631
2.4598541259765625
2.4598300457000732
2.459805965423584
2.459782361984253
2.459758758544922
2.459735155105591
2.459712028503418
2.459688901901245
2.459666

In [207]:
for i in range(5):
    ans = []
    w = 0
    while True:
        xenc = F.one_hot(torch.tensor([w]), num_classes = 27).float()
        logits = xenc @ W
        probs = logits.exp()
        probs = probs / probs.sum(1, keepdims = True)
        w = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
        ans.append(itos[w])
        if (w == 0):
            break
    print(''.join(ans))

n.
de.
meeo.
s.
nyadenafadyaran.
